# Text-to-NoSQL System Demo

This notebook demonstrates the SMART framework for translating natural language queries to MongoDB queries.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

from smart.smart_pipeline import create_smart_framework
from utils.sample_generator import SampleDataGenerator
import json

print("Imports successful!")

## 2. Generate Sample Data

In [ ]:
generator = SampleDataGenerator()
data = generator.generate_ecommerce_database()

print(f"Generated {len(data['databases'])} collections")
print(f"Generated {len(data['queries'])} sample queries")
print("\nCollections:")
for coll_name in data['databases'].keys():
    print(f"  - {coll_name}")

## 3. Initialize SMART Framework

In [ ]:
# Create framework
framework = create_smart_framework('../configs/config.yaml')

# Connect to MongoDB (optional - needed for execution)
connected = framework.connect_mongodb()
if connected:
    print("✓ Connected to MongoDB")
else:
    print("✗ MongoDB not available - will skip execution steps")

## 4. Load Schemas

In [ ]:
# Load schemas from sample data
framework.load_schemas(data['schema_summary'])

print("Loaded schemas:")
for coll, fields in framework.schemas.items():
    print(f"\n{coll}:")
    print(f"  Fields: {', '.join(fields)}")

## 5. Index Training Examples (for RAG)

In [ ]:
# Index the sample queries for RAG retrieval
framework.index_training_examples(data['queries'])
print(f"Indexed {len(data['queries'])} training examples for RAG")

## 6. Translate Queries - Step by Step

Let's see each step of the SMART pipeline in action.

In [ ]:
# Example natural language query
nlq = "Find all products in the Electronics category"

print(f"Natural Language Query: {nlq}")
print("\n" + "="*60)

# Translate
result = framework.translate(nlq, use_execution_optimization=False)

# Display steps
print("\nStep 1: Schema Prediction")
print(f"  Collection: {result['steps']['schema_prediction']['collection']}")
print(f"  Fields: {result['steps']['schema_prediction']['fields']}")

print("\nStep 2: Initial Query Generation")
print(f"  {result['steps']['initial_query']}")

print("\nStep 3: RAG-Refined Query")
print(f"  {result['steps']['refined_query']}")

print("\nFinal Query:")
print(f"  {result['final_query']}")

## 7. Try Multiple Queries

In [ ]:
test_queries = [
    "Count the total number of products",
    "Find customers in New York",
    "What is the average price of all products?",
    "Show me products that cost less than $100"
]

for i, nlq in enumerate(test_queries, 1):
    print(f"\n{i}. {nlq}")
    result = framework.translate(nlq, use_execution_optimization=False)
    print(f"   → {result['final_query']}")

## 8. With Execution (if MongoDB connected)

In [ ]:
if connected:
    # Setup sample data in MongoDB
    for coll_name, coll_data in data['databases'].items():
        framework.mongo_client.create_collection(coll_name, coll_data['documents'])
    print("Sample data loaded into MongoDB")
    
    # Now try with execution
    nlq = "Find all products in the Electronics category"
    result = framework.translate(nlq, use_execution_optimization=True)
    
    print(f"\nQuery: {nlq}")
    print(f"Generated: {result['final_query']}")
    print(f"Success: {result['success']}")
    
    if result.get('results'):
        print(f"\nResults ({len(result['results'])} documents):")
        print(json.dumps(result['results'], indent=2))
else:
    print("MongoDB not connected - skipping execution demo")

## 9. Evaluation Example

In [ ]:
from evaluation.evaluator import Evaluator

evaluator = Evaluator(framework.mongo_client if connected else None)

# Example evaluation
predicted = 'db.Products.find({"category": "Electronics"});'
gold = 'db.Products.find({"category": "Electronics"});'

metrics = evaluator.evaluate_single(predicted, gold, compute_execution=connected)

print("Evaluation Metrics:")
print(f"  Exact Match: {metrics['exact_match']}")
print(f"  Collection Match: {metrics['component_match']['collection_match']}")
print(f"  Operation Match: {metrics['component_match']['operation_match']}")

if 'execution_accuracy' in metrics:
    print(f"  Execution Accuracy: {metrics['execution_accuracy']}")

## 10. Cleanup

In [ ]:
# Disconnect from MongoDB
if connected:
    framework.disconnect()
    print("Disconnected from MongoDB")

## Summary

This notebook demonstrated:

1. ✅ Setting up the SMART framework
2. ✅ Loading schemas and training examples
3. ✅ Step-by-step query translation pipeline
4. ✅ Schema prediction → Query generation → RAG refinement → Execution
5. ✅ Evaluation metrics

**Next Steps:**
- Try with your own MongoDB databases
- Add more training examples for better RAG performance
- Fine-tune SLMs for production use
- Integrate with your application